# 02 · Check exact correction against dense replay

The C ring-buffer implementation and dense NumPy oracle compute the same signed INT8 scores with independently written implementations. The tests compare every retained class score both before and after a correction transaction. The native counters record kernel sites and MACs for the broad replay grid.

Set `TINYML_NOTEBOOK_REBUILD=1` to run the native corpus from its bundled held-out clips and compare every regenerated row.

In [1]:
import json
import os
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "study.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from tinyml.config import DATASETS, RESULTS, SEEDS, run_dir

REBUILD = os.environ.get("TINYML_NOTEBOOK_REBUILD") == "1"
DATA = Path(os.environ.get("TINYML_DATA_DIR", ROOT / "data")).resolve()
WORKERS = os.environ.get("TINYML_NOTEBOOK_WORKERS", "2")
SCRATCH = ROOT / "build" / "notebooks"

def run(*args):
    command = [sys.executable, *map(str, args)]
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout + result.stderr)
        result.check_returncode()
    return result.stdout + result.stderr

print("Execution mode:", "full reproduction" if REBUILD else "inspect archived outputs")
print("Models:", len(DATASETS) * len(SEEDS), "| seeds:", ", ".join(map(str, SEEDS)))

Execution mode: full reproduction
Models: 6 | seeds: 13, 37, 73


In [2]:
import shutil
from tinyml.analysis import paired, principal
from tinyml.protocol import MODES

print(run("-m", "unittest", "discover", "-s", "tests", "-v"), end="")
if REBUILD:
    for dataset in DATASETS:
        for seed in SEEDS:
            source = run_dir(RESULTS, dataset, seed)
            target = run_dir(SCRATCH, dataset, seed)
            target.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source / "int8.npz", target / "int8.npz")
    print(run("-m", "tinyml.evaluate", "--backend", "native", "--results", SCRATCH,
              "--build", SCRATCH / "host_build", "--workers", WORKERS), end="")
    for dataset in DATASETS:
        for seed in SEEDS:
            name = "native_events.csv.gz"
            pd.testing.assert_frame_equal(
                pd.read_csv(run_dir(SCRATCH, dataset, seed) / name),
                pd.read_csv(run_dir(RESULTS, dataset, seed) / name),
            )
    print("All six regenerated native event tables match the archived rows.")

test_dense_rounding_and_saturation (test_bit_exact.ExactFirmwareTests.test_dense_rounding_and_saturation) ... ok
test_fixture_ids_and_shapes (test_bit_exact.ExactFirmwareTests.test_fixture_ids_and_shapes) ... ok
test_rejected_updates_are_atomic_and_identical_updates_are_noops (test_bit_exact.ExactFirmwareTests.test_rejected_updates_are_atomic_and_identical_updates_are_noops) ... ok
test_ring_wrap_repeated_corrections_and_future_predictions (test_bit_exact.ExactFirmwareTests.test_ring_wrap_repeated_corrections_and_future_predictions) ... ok

----------------------------------------------------------------------
Ran 4 tests in 15.076s

OK


native wisdm/37 horizon=48: 3,096 cases × 8 methods
native wisdm/37 horizon=8: 432 cases × 8 methods
native wisdm/37 horizon=24: 432 cases × 8 methods
native wisdm/13 horizon=48: 3,096 cases × 8 methods
native wisdm/13 horizon=8: 432 cases × 8 methods
native wisdm/13 horizon=24: 432 cases × 8 methods
native mhealth/13 horizon=48: 2,064 cases × 8 methods
native mhealth/13 horizon=8: 288 cases × 8 methods
native mhealth/13 horizon=24: 288 cases × 8 methods
native wisdm/73 horizon=48: 3,096 cases × 8 methods
native wisdm/73 horizon=8: 432 cases × 8 methods
native mhealth/37 horizon=48: 2,064 cases × 8 methods
native wisdm/73 horizon=24: 432 cases × 8 methods
native mhealth/37 horizon=8: 288 cases × 8 methods
native mhealth/37 horizon=24: 288 cases × 8 methods
native mhealth/73 horizon=48: 2,064 cases × 8 methods
native mhealth/73 horizon=8: 288 cases × 8 methods
native mhealth/73 horizon=24: 288 cases × 8 methods


All six regenerated native event tables match the archived rows.


In [3]:
frames = []
for dataset in DATASETS:
    for seed in SEEDS:
        frame = pd.read_csv(run_dir(RESULTS, dataset, seed) / "native_events.csv.gz")
        assert set(frame["mode"]) == set(MODES)
        assert frame["mismatches"].sum() == 0
        frames.append(frame)
native = pd.concat(frames, ignore_index=True)
checks = native.groupby(["dataset", "horizon"]).agg(
    runs=("mode", "size"), checked_scores=("score_pairs_checked", "sum"),
    mismatches=("mismatches", "sum"),
)
display(checks)

runs  checked_scores  mismatches
dataset horizon                                   
mhealth 8         6912         1492992           0
        24        6912         4147200           0
        48       49536        58254336           0
wisdm   8        10368         3359232           0
        24       10368         9331200           0
        48       74304       131072256           0

In [4]:
host = principal(native)
modes = ["tap", "tap_equal", "adaptive", "serial"]
effects = pd.concat([paired(host, "kernel_macs", mode) for mode in modes])
values = effects.pivot_table(index=["dataset", "mode"], columns="seed",
                             values="reduction_pct", aggfunc="median")
display(values.round(2))
print("Values are median paired kernel-MAC savings (%) versus temporal replay.")

seed                  13     37     73
dataset mode                          
mhealth adaptive   30.05  29.84  29.84
        serial      3.30   3.30   3.30
        tap        25.89  25.89  25.89
        tap_equal  32.42  29.84  30.05
wisdm   adaptive   27.14  26.15  25.64
        serial      3.08   2.15   0.00
        tap        25.59  25.59  25.59
        tap_equal  27.94  27.22  26.52

Values are median paired kernel-MAC savings (%) versus temporal replay.


Native MACs count convolution work. Check the next notebook for emulated ARM instruction counts, which include mask traversal and equality overhead.